# GPU test of the own training engine

Builds `wav-transformer` with `--features gpu-engine` (cuBLAS + own CUDA kernels compiled at run time with NVRTC, no
nvcc needed) and runs the tests that compare the CUDA backend with the CPU backend (loss and every gradient, f32 and
f16). *Session options*: **Accelerator GPU T4**, **Internet on**.


In [ ]:
REPO = "https://github.com/RodionKovalenko/rustCrates"
BRANCH = "feature/neural-network-rust"
SOURCE_DIR = "/tmp/rustCrates"

import glob, os, subprocess, sys

def run(cmd, **kw):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kw)
    print(p.stdout[-20000:], flush=True)
    if p.returncode != 0:
        raise SystemExit(f"command failed with exit code {p.returncode}")
    return p.stdout

run("nvidia-smi")
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]
if not os.path.exists(SOURCE_DIR):
    run(f"git clone --depth 50 --branch {BRANCH} {REPO} {SOURCE_DIR}")
run("git log -1 --oneline", cwd=SOURCE_DIR)

# cudarc loads libcuda, cuBLAS and NVRTC at run time: make the toolkit libraries findable.
libs = sorted({os.path.dirname(p) for p in glob.glob("/usr/local/cuda*/**/libcublas.so*", recursive=True) + glob.glob("/usr/local/cuda*/**/libnvrtc.so*", recursive=True)})
os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("library dirs:", libs)


In [ ]:
out = run("cargo test --release --features gpu-engine --lib gpu_engine -- --nocapture --test-threads=1", cwd=SOURCE_DIR)
open("/kaggle/working/engine_test.log", "w").write(out)
